In [ ]:
#unnecessary ipynb need to delete later

import os
import pandas as pd
import numpy as np

# -------------------------------------------------------------------------
# Step 2: Completeness & Missing Value Audit
# -------------------------------------------------------------------------

# 1. Load the deduplicated clean dataset
CLEAN_DATA_PATH = r"C:\Project\data\processed\clean_prices_festivals_2010_2016.csv"

# Fallback in case path doesn't exist yet
if not os.path.exists(CLEAN_DATA_PATH):
    RAW_PATH = r"C:\Project\data\raw\flower_prices_with_festivals_2010_2016.csv"
    df_raw = pd.read_csv(RAW_PATH)
    df_raw['DATE'] = pd.to_datetime(df_raw['DATE'])
    df = df_raw.groupby('DATE').agg({
        'festival_name': lambda x: ' | '.join(sorted(x.dropna().astype(str).unique())) if len(x.dropna()) > 0 else np.nan,
        'tithi': lambda x: ' | '.join(sorted(x.dropna().astype(str).unique())) if len(x.dropna()) > 0 else np.nan,
        'hindu_month': lambda x: ' | '.join(sorted(x.dropna().astype(str).unique())) if len(x.dropna()) > 0 else np.nan,
        'price': 'first'
    }).reset_index()
    df['is_festival'] = df['festival_name'].notna().astype(int)
else:
    df = pd.read_csv(CLEAN_DATA_PATH)
    df['DATE'] = pd.to_datetime(df['DATE'])

total_rows = len(df)

# 2. Perform completeness audit across all features
audit_metrics = []

for column in df.columns:
    present_count = df[column].notnull().sum()
    missing_count = df[column].isnull().sum()
    missing_pct = (missing_count / total_rows) * 100
    completeness_pct = (present_count / total_rows) * 100
    
    audit_metrics.append({
        'Column Name': column,
        'Data Type': str(df[column].dtype),
        'Present Count': present_count,
        'Missing Count': missing_count,
        'Missing %': f"{missing_pct:.2f}%",
        'Completeness %': f"{completeness_pct:.2f}%"
    })

audit_df = pd.DataFrame(audit_metrics)

# -------------------------------------------------------------------------
# Output Summary Report
# -------------------------------------------------------------------------
print("--- STEP 2: COMPLETENESS & MISSING VALUE AUDIT REPORT ---")
print(f"Total Dataset Rows Analyzed: {total_rows}\n")
print(audit_df.to_string(index=False))

print("\n--- KEY AUDIT INSIGHTS ---")
print(f"• Target Variable ('price'): 100% complete ({df['price'].notnull().sum()}/{total_rows} entries present). Zero imputation needed.")
print(f"• Primary Key ('DATE'): 100% complete across all 2,557 daily calendar records.")
print(f"• Sparse Event Features ('festival_name'): Missing values (94.80%) represent non-festival ordinary trading days.")

--- STEP 2: COMPLETENESS & MISSING VALUE AUDIT REPORT ---
Total Dataset Rows Analyzed: 2557

  Column Name      Data Type  Present Count  Missing Count Missing % Completeness %
         DATE datetime64[us]           2557              0     0.00%        100.00%
festival_name            str            133           2424    94.80%          5.20%
        tithi            str            119           2438    95.35%          4.65%
  hindu_month            str            126           2431    95.07%          4.93%
        price          int64           2557              0     0.00%        100.00%
  is_festival          int64           2557              0     0.00%        100.00%

--- KEY AUDIT INSIGHTS ---
• Target Variable ('price'): 100% complete (2557/2557 entries present). Zero imputation needed.
• Primary Key ('DATE'): 100% complete across all 2,557 daily calendar records.
• Sparse Event Features ('festival_name'): Missing values (94.80%) represent non-festival ordinary trading days.
